# Module 3: Sentiment Aggregation Across Sources

## Learning Objectives

By completing this notebook, you will:
1. Aggregate sentiment from multiple news sources and analysts
2. Implement weighting schemes based on source credibility and relevance
3. Apply time decay functions to prioritize recent sentiment
4. Build composite sentiment indicators for trading decisions
5. Handle conflicting signals and measure consensus

## Prerequisites

- Completed Module 3, Notebook 1 (News Sentiment Analysis)
- Understanding of sentiment scoring
- Basic statistics (weighted averages, standard deviation)
- LLM API access

## Estimated Time: 75-95 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Completed Module 3, Notebook 1 (News Sentiment Analysis)",
    "Understanding of sentiment scoring",
    "Basic statistics (weighted averages, standard deviation)",
    "LLM API access"
])

## 1. The Challenge of Multi-Source Sentiment

**Problem:** Different sources give conflicting signals.

**Example Scenario:**
- **Bloomberg:** "Crude oil inventories fall sharply" (Bullish)
- **Reuters:** "OPEC hints at production increase" (Bearish)
- **Twitter:** "Refinery fire disrupts gasoline supply" (Bullish)
- **Analyst Report:** "Demand concerns overshadow supply tightness" (Bearish)

**Questions:**
- How do we combine these?
- Which sources are more reliable?
- How do we handle recency?
- What's the overall market sentiment?

**Solution Framework:**
1. Source-specific weights (credibility)
2. Time decay (recency)
3. Confidence-based weighting
4. Consensus measurement (agreement)
5. Outlier detection

In [ ]:
# Purpose: Setup imports and sample sentiment data
# Key Concept: Aggregation requires structured sentiment data

import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Tuple
from pathlib import Path

import pandas as pd
import numpy as np
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# LLM imports (for advanced features)
import openai
from anthropic import Anthropic

# Configuration
load_dotenv()
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 7)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None

print("✅ Setup complete")
print(f"   LLM: {'OpenAI' if openai_client else 'Claude' if anthropic_client else 'None'}")

In [ ]:
# Purpose: Create sample sentiment data from multiple sources
# Key Concept: Real-world data has varying quality and timing

# Sample sentiment data
np.random.seed(42)
now = datetime.now()

sentiment_data = [
    {
        'source': 'Bloomberg',
        'timestamp': now - timedelta(hours=2),
        'headline': 'Crude oil inventories decline sharply',
        'sentiment_score': 1,
        'confidence': 0.85,
        'category': 'news'
    },
    {
        'source': 'Reuters',
        'timestamp': now - timedelta(hours=4),
        'headline': 'OPEC considers production increase',
        'sentiment_score': -1,
        'confidence': 0.75,
        'category': 'news'
    },
    {
        'source': 'Twitter',
        'timestamp': now - timedelta(hours=1),
        'headline': 'Major refinery fire disrupts supply',
        'sentiment_score': 2,
        'confidence': 0.60,
        'category': 'social'
    },
    {
        'source': 'Goldman Sachs Research',
        'timestamp': now - timedelta(hours=12),
        'headline': 'Demand concerns overshadow supply tightness',
        'sentiment_score': -1,
        'confidence': 0.90,
        'category': 'analyst'
    },
    {
        'source': 'CNBC',
        'timestamp': now - timedelta(hours=3),
        'headline': 'Mixed signals in oil market',
        'sentiment_score': 0,
        'confidence': 0.70,
        'category': 'news'
    },
    {
        'source': 'EIA Report',
        'timestamp': now - timedelta(hours=6),
        'headline': 'Weekly inventory draw exceeds expectations',
        'sentiment_score': 1,
        'confidence': 0.95,
        'category': 'official'
    },
    {
        'source': 'MarketWatch',
        'timestamp': now - timedelta(hours=8),
        'headline': 'Oil prices under pressure from strong dollar',
        'sentiment_score': -1,
        'confidence': 0.80,
        'category': 'news'
    },
    {
        'source': 'JP Morgan',
        'timestamp': now - timedelta(hours=24),
        'headline': 'Bullish outlook on crude fundamentals',
        'sentiment_score': 2,
        'confidence': 0.88,
        'category': 'analyst'
    }
]

# Convert to DataFrame
df = pd.DataFrame(sentiment_data)

print("Sample Sentiment Data:")
print("="*80)
print(df[['source', 'timestamp', 'sentiment_score', 'confidence', 'category']])
print("\n" + "="*80)
print(f"Total sources: {len(df)}")
print(f"Categories: {df['category'].unique()}")
print(f"Time range: {df['timestamp'].min()} to {df['timestamp'].max()}")

## 2. Source-Based Weighting

Not all sources are equal. We assign weights based on:

**Credibility Factors:**
- **Official data** (EIA, IEA): Highest weight (1.0)
- **Major analyst firms** (Goldman, JP Morgan): High weight (0.9)
- **Established news** (Bloomberg, Reuters): Medium-high weight (0.8)
- **General news** (CNBC, MarketWatch): Medium weight (0.7)
- **Social media**: Lower weight (0.5)

**Additional Factors:**
- Track record accuracy
- Domain expertise
- Editorial standards

In [ ]:
# Purpose: Implement source-based weighting system
# Key Concept: Weight by source credibility and category

# Define source weights
SOURCE_WEIGHTS = {
    'EIA Report': 1.0,
    'IEA': 1.0,
    'Goldman Sachs Research': 0.9,
    'JP Morgan': 0.9,
    'Morgan Stanley': 0.9,
    'Bloomberg': 0.85,
    'Reuters': 0.85,
    'CNBC': 0.7,
    'MarketWatch': 0.7,
    'Twitter': 0.5,
    'Reddit': 0.4
}

# Category-based weights (fallback if source not in dictionary)
CATEGORY_WEIGHTS = {
    'official': 1.0,
    'analyst': 0.85,
    'news': 0.75,
    'social': 0.5
}

def get_source_weight(source: str, category: str) -> float:
    """
    Get weight for a source.
    
    Parameters:
    -----------
    source : str
        Source name
    category : str
        Source category
        
    Returns:
    --------
    float
        Weight (0-1)
    """
    # Try exact source match first
    if source in SOURCE_WEIGHTS:
        return SOURCE_WEIGHTS[source]
    
    # Fall back to category
    return CATEGORY_WEIGHTS.get(category, 0.6)

# Add source weights to DataFrame
df['source_weight'] = df.apply(lambda row: get_source_weight(row['source'], row['category']), axis=1)

print("Source Weights:")
print("="*80)
print(df[['source', 'category', 'source_weight']].sort_values('source_weight', ascending=False))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
df_sorted = df.sort_values('source_weight', ascending=True)
colors = df_sorted['category'].map({
    'official': 'green',
    'analyst': 'blue',
    'news': 'orange',
    'social': 'red'
})

ax.barh(df_sorted['source'], df_sorted['source_weight'], color=colors, alpha=0.7)
ax.set_xlabel('Source Weight', fontsize=12)
ax.set_title('Source Credibility Weights', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='green', alpha=0.7, label='Official'),
    Patch(facecolor='blue', alpha=0.7, label='Analyst'),
    Patch(facecolor='orange', alpha=0.7, label='News'),
    Patch(facecolor='red', alpha=0.7, label='Social')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

## 3. Time Decay Functions

Recent sentiment matters more than old sentiment.

**Time Decay Approaches:**

1. **Exponential Decay**: `weight = exp(-λ * hours_old)`
   - Half-life: time for weight to drop to 0.5
   - Fast decay for intraday trading
   - Typical half-life: 12-48 hours

2. **Linear Decay**: `weight = max(0, 1 - hours_old / max_hours)`
   - Simple and interpretable
   - Hard cutoff at max_hours

3. **Step Function**: `weight = 1.0 if hours_old < threshold else 0.5`
   - Discrete time buckets
   - Less smooth but simple

In [ ]:
# Purpose: Implement time decay functions
# Key Concept: Recent sentiment gets higher weight

def exponential_decay(hours_old: float, half_life_hours: float = 24.0) -> float:
    """
    Calculate exponential time decay weight.
    
    Parameters:
    -----------
    hours_old : float
        Hours since timestamp
    half_life_hours : float
        Time for weight to decay to 0.5
        
    Returns:
    --------
    float
        Decay weight (0-1)
    """
    lambda_decay = np.log(2) / half_life_hours
    return np.exp(-lambda_decay * hours_old)

def linear_decay(hours_old: float, max_hours: float = 72.0) -> float:
    """
    Calculate linear time decay weight.
    
    Parameters:
    -----------
    hours_old : float
        Hours since timestamp
    max_hours : float
        Hours at which weight reaches 0
        
    Returns:
    --------
    float
        Decay weight (0-1)
    """
    return max(0.0, 1.0 - hours_old / max_hours)

def step_decay(hours_old: float, thresholds: List[Tuple[float, float]] = None) -> float:
    """
    Calculate step function time decay.
    
    Parameters:
    -----------
    hours_old : float
        Hours since timestamp
    thresholds : List[Tuple[float, float]]
        List of (max_hours, weight) tuples
        
    Returns:
    --------
    float
        Decay weight
    """
    if thresholds is None:
        thresholds = [(12, 1.0), (24, 0.7), (48, 0.4), (72, 0.2)]
    
    for max_hours, weight in thresholds:
        if hours_old <= max_hours:
            return weight
    
    return 0.1  # Very old data

# Calculate hours old for each sentiment
df['hours_old'] = df['timestamp'].apply(lambda x: (now - x).total_seconds() / 3600)

# Apply different decay functions
df['exp_decay'] = df['hours_old'].apply(lambda x: exponential_decay(x, half_life_hours=24))
df['linear_decay'] = df['hours_old'].apply(lambda x: linear_decay(x, max_hours=72))
df['step_decay'] = df['hours_old'].apply(step_decay)

print("Time Decay Comparison:")
print("="*80)
print(df[['source', 'hours_old', 'exp_decay', 'linear_decay', 'step_decay']].round(3))

# Visualize decay functions
hours_range = np.linspace(0, 72, 100)
exp_values = [exponential_decay(h, 24) for h in hours_range]
linear_values = [linear_decay(h, 72) for h in hours_range]
step_values = [step_decay(h) for h in hours_range]

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hours_range, exp_values, label='Exponential (half-life=24h)', linewidth=2)
ax.plot(hours_range, linear_values, label='Linear (max=72h)', linewidth=2)
ax.plot(hours_range, step_values, label='Step Function', linewidth=2)

# Mark actual data points
ax.scatter(df['hours_old'], df['exp_decay'], color='red', s=100, alpha=0.6, zorder=5)

ax.set_xlabel('Hours Old', fontsize=12)
ax.set_ylabel('Decay Weight', fontsize=12)
ax.set_title('Time Decay Functions', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

### 💡 Exercise 3.1: Weighted Sentiment Aggregation

**Task:** Implement a complete weighted sentiment aggregation function.

**Requirements:**
1. Combine source weight, time decay, and confidence
2. Calculate final weight: `final_weight = source_weight * time_decay * confidence`
3. Compute weighted average sentiment: `Σ(sentiment * weight) / Σ(weight)`
4. Calculate weighted standard deviation (measure of consensus)
5. Determine if sentiment is statistically significant

**Function Signature:**
```python
def aggregate_sentiment(df: pd.DataFrame, 
                       decay_function: str = 'exponential',
                       half_life: float = 24.0) -> Dict[str, float]:
    """
    Returns:
        dict with keys: weighted_sentiment, std_dev, num_sources, confidence
    """
```

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------

def aggregate_sentiment(df: pd.DataFrame, 
                       decay_function: str = 'exponential',
                       half_life: float = 24.0) -> Dict[str, Any]:
    """
    Aggregate sentiment with source, time, and confidence weighting.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with sentiment data
    decay_function : str
        Time decay function: 'exponential', 'linear', or 'step'
    half_life : float
        Half-life for exponential decay (hours)
        
    Returns:
    --------
    dict
        Aggregated sentiment metrics
    """
    # Calculate time decay
    if decay_function == 'exponential':
        df['time_decay'] = df['hours_old'].apply(lambda x: exponential_decay(x, half_life))
    elif decay_function == 'linear':
        df['time_decay'] = df['hours_old'].apply(lambda x: linear_decay(x, max_hours=72))
    elif decay_function == 'step':
        df['time_decay'] = df['hours_old'].apply(step_decay)
    else:
        df['time_decay'] = 1.0
    
    # Calculate combined weight
    df['final_weight'] = df['source_weight'] * df['time_decay'] * df['confidence']
    
    # Weighted average sentiment
    total_weight = df['final_weight'].sum()
    if total_weight == 0:
        return {'error': 'Total weight is zero'}
    
    weighted_sentiment = (df['sentiment_score'] * df['final_weight']).sum() / total_weight
    
    # Weighted standard deviation
    weighted_variance = ((df['sentiment_score'] - weighted_sentiment) ** 2 * df['final_weight']).sum() / total_weight
    weighted_std = np.sqrt(weighted_variance)
    
    # Consensus measure (inverse of std)
    consensus = 1.0 / (1.0 + weighted_std)
    
    # Overall confidence (average of weights)
    overall_confidence = df['final_weight'].mean()
    
    # Determine signal
    if weighted_sentiment >= 0.5:
        signal = "BULLISH"
    elif weighted_sentiment <= -0.5:
        signal = "BEARISH"
    else:
        signal = "NEUTRAL"
    
    return {
        'weighted_sentiment': weighted_sentiment,
        'std_dev': weighted_std,
        'consensus': consensus,
        'num_sources': len(df),
        'total_weight': total_weight,
        'signal': signal,
        'confidence': overall_confidence
    }

# Test aggregation
print("Sentiment Aggregation Results\n")
print("="*80)

for decay_func in ['exponential', 'linear', 'step']:
    result = aggregate_sentiment(df.copy(), decay_function=decay_func, half_life=24)
    
    print(f"\n[{decay_func.upper()} Decay]")
    print("-"*80)
    print(f"  Weighted Sentiment: {result['weighted_sentiment']:+.3f}")
    print(f"  Signal: {result['signal']}")
    print(f"  Standard Deviation: {result['std_dev']:.3f}")
    print(f"  Consensus: {result['consensus']:.3f} (higher = more agreement)")
    print(f"  Sources: {result['num_sources']}")
    print(f"  Overall Confidence: {result['confidence']:.3f}")

print("\n" + "="*80)

## 4. Consensus and Disagreement Metrics

**Why Consensus Matters:**
- High consensus → Strong signal
- Low consensus → Mixed/uncertain market
- Extreme disagreement → Potential inflection point

**Metrics:**

1. **Standard Deviation**: Spread of sentiment scores
2. **Sentiment Range**: Max - Min sentiment
3. **Directional Agreement**: % of sources agreeing with aggregate direction
4. **Outlier Detection**: Sources far from consensus

In [ ]:
# Purpose: Calculate consensus and disagreement metrics
# Key Concept: Measure market uncertainty and conviction

def calculate_consensus_metrics(df: pd.DataFrame, weighted_sentiment: float) -> Dict[str, Any]:
    """
    Calculate consensus and disagreement metrics.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Sentiment data with weights
    weighted_sentiment : float
        Aggregate weighted sentiment
        
    Returns:
    --------
    dict
        Consensus metrics
    """
    # Directional agreement
    aggregate_direction = np.sign(weighted_sentiment)
    agreement_count = (np.sign(df['sentiment_score']) == aggregate_direction).sum()
    directional_agreement = agreement_count / len(df)
    
    # Sentiment range
    sentiment_range = df['sentiment_score'].max() - df['sentiment_score'].min()
    
    # Identify outliers (>2 std from mean)
    mean_sentiment = df['sentiment_score'].mean()
    std_sentiment = df['sentiment_score'].std()
    df['is_outlier'] = np.abs(df['sentiment_score'] - mean_sentiment) > (2 * std_sentiment)
    outliers = df[df['is_outlier']]
    
    # Bullish/Bearish split
    bullish_count = (df['sentiment_score'] > 0).sum()
    bearish_count = (df['sentiment_score'] < 0).sum()
    neutral_count = (df['sentiment_score'] == 0).sum()
    
    # Consensus strength (0-1, 1 = perfect agreement)
    max_agreement = max(bullish_count, bearish_count, neutral_count)
    consensus_strength = max_agreement / len(df)
    
    return {
        'directional_agreement': directional_agreement,
        'sentiment_range': sentiment_range,
        'num_outliers': len(outliers),
        'outlier_sources': outliers['source'].tolist() if len(outliers) > 0 else [],
        'bullish_count': int(bullish_count),
        'bearish_count': int(bearish_count),
        'neutral_count': int(neutral_count),
        'consensus_strength': consensus_strength
    }

# Calculate consensus metrics
agg_result = aggregate_sentiment(df.copy(), decay_function='exponential', half_life=24)
consensus_metrics = calculate_consensus_metrics(df, agg_result['weighted_sentiment'])

print("Consensus Analysis\n")
print("="*80)
print(f"Aggregate Sentiment: {agg_result['weighted_sentiment']:+.3f} ({agg_result['signal']})")
print("-"*80)
print(f"Directional Agreement: {consensus_metrics['directional_agreement']:.1%}")
print(f"Consensus Strength: {consensus_metrics['consensus_strength']:.1%}")
print(f"Sentiment Range: {consensus_metrics['sentiment_range']}")
print(f"\nSource Distribution:")
print(f"  Bullish: {consensus_metrics['bullish_count']}")
print(f"  Bearish: {consensus_metrics['bearish_count']}")
print(f"  Neutral: {consensus_metrics['neutral_count']}")
print(f"\nOutliers: {consensus_metrics['num_outliers']}")
if consensus_metrics['outlier_sources']:
    print(f"  Sources: {', '.join(consensus_metrics['outlier_sources'])}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Sentiment distribution
sentiment_counts = [consensus_metrics['bullish_count'], 
                   consensus_metrics['neutral_count'],
                   consensus_metrics['bearish_count']]
labels = ['Bullish', 'Neutral', 'Bearish']
colors_pie = ['green', 'gray', 'red']

ax1.pie(sentiment_counts, labels=labels, colors=colors_pie, autopct='%1.1f%%', startangle=90)
ax1.set_title('Sentiment Distribution', fontsize=14, fontweight='bold')

# Individual source sentiments
df_sorted = df.sort_values('final_weight', ascending=True)
colors_bar = df_sorted['sentiment_score'].apply(
    lambda x: 'green' if x > 0 else 'red' if x < 0 else 'gray'
)

ax2.barh(df_sorted['source'], df_sorted['sentiment_score'], color=colors_bar, alpha=0.7)
ax2.axvline(x=agg_result['weighted_sentiment'], color='blue', linestyle='--', linewidth=2, label='Weighted Avg')
ax2.axvline(x=0, color='black', linestyle='-', alpha=0.3)
ax2.set_xlabel('Sentiment Score', fontsize=12)
ax2.set_title('Individual Source Sentiment', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n" + "="*80)

### 💡 Exercise 4.1: Dynamic Weighting System

**Task:** Build a dynamic weighting system that adapts based on source performance.

**Concept:** Track which sources have been most accurate and adjust weights accordingly.

**Requirements:**
1. Create historical accuracy tracker for each source
2. Calculate accuracy score (how often sentiment matched price movement)
3. Adjust source weights based on recent accuracy
4. Implement exponential moving average for accuracy
5. Compare static vs dynamic weighting results

**Hint:** Accuracy can be measured by correlation between sentiment and next-day price change

In [ ]:
# YOUR CODE HERE
# ---------------





In [ ]:
# SOLUTION
# --------

# Simulate historical accuracy data
np.random.seed(42)
source_accuracy = {
    'EIA Report': 0.85,
    'Goldman Sachs Research': 0.78,
    'JP Morgan': 0.82,
    'Bloomberg': 0.72,
    'Reuters': 0.70,
    'CNBC': 0.60,
    'MarketWatch': 0.58,
    'Twitter': 0.52
}

def calculate_dynamic_weights(source_weights: Dict[str, float],
                            source_accuracy: Dict[str, float],
                            accuracy_factor: float = 0.5) -> Dict[str, float]:
    """
    Calculate dynamic weights based on historical accuracy.
    
    Parameters:
    -----------
    source_weights : dict
        Base source weights
    source_accuracy : dict
        Historical accuracy (0-1) for each source
    accuracy_factor : float
        How much to weight accuracy vs base weight (0-1)
        
    Returns:
    --------
    dict
        Adjusted weights
    """
    dynamic_weights = {}
    
    for source, base_weight in source_weights.items():
        accuracy = source_accuracy.get(source, 0.5)
        
        # Blend base weight with accuracy
        adjusted_weight = (1 - accuracy_factor) * base_weight + accuracy_factor * accuracy
        dynamic_weights[source] = adjusted_weight
    
    return dynamic_weights

# Calculate dynamic weights for our data
base_weights = {source: get_source_weight(source, cat) 
               for source, cat in zip(df['source'], df['category'])}

dynamic_weights = calculate_dynamic_weights(base_weights, source_accuracy, accuracy_factor=0.5)

# Add dynamic weights to DataFrame
df['dynamic_weight'] = df['source'].map(dynamic_weights)

# Compare static vs dynamic aggregation
print("Static vs Dynamic Weighting Comparison\n")
print("="*80)

# Static
static_result = aggregate_sentiment(df.copy(), decay_function='exponential')
print("[Static Weights]")
print(f"  Weighted Sentiment: {static_result['weighted_sentiment']:+.3f}")
print(f"  Signal: {static_result['signal']}")
print(f"  Std Dev: {static_result['std_dev']:.3f}")

# Dynamic (replace source_weight with dynamic_weight)
df_dynamic = df.copy()
df_dynamic['source_weight'] = df_dynamic['dynamic_weight']
dynamic_result = aggregate_sentiment(df_dynamic, decay_function='exponential')
print("\n[Dynamic Weights (accuracy-adjusted)]")
print(f"  Weighted Sentiment: {dynamic_result['weighted_sentiment']:+.3f}")
print(f"  Signal: {dynamic_result['signal']}")
print(f"  Std Dev: {dynamic_result['std_dev']:.3f}")

print("\n[Weight Adjustments]")
print("-"*80)
weight_comparison = pd.DataFrame({
    'Source': df['source'],
    'Static Weight': df['source_weight'],
    'Accuracy': df['source'].map(source_accuracy),
    'Dynamic Weight': df['dynamic_weight'],
    'Change': df['dynamic_weight'] - df['source_weight']
})
print(weight_comparison.round(3))

print("\n" + "="*80)
print("Interpretation:")
print("  • Dynamic weights reward consistently accurate sources")
print("  • Can shift aggregate sentiment if high-accuracy sources disagree with base")
print("  • Requires ongoing tracking of source accuracy")

## Summary

### Key Takeaways

1. **Source Weighting is Critical** - Not all news sources are equally reliable

2. **Time Decay Prioritizes Recent** - Exponential decay with 24-48hr half-life works well

3. **Combine Multiple Factors** - Source credibility × Time decay × Confidence

4. **Consensus Matters** - Low consensus = uncertain market, high consensus = strong signal

5. **Dynamic Weights Improve Over Time** - Track accuracy and adjust weights

### Skills Gained

✅ Implementing source-based weighting schemes  
✅ Applying time decay functions  
✅ Calculating weighted sentiment aggregates  
✅ Measuring consensus and disagreement  
✅ Building dynamic weighting systems  
✅ Visualizing sentiment distributions  

### What's Next

In **Module 3, Notebook 3** (`03_signal_pipeline.ipynb`), we'll learn:
- Converting sentiment to trading signals
- Signal strength and confidence thresholds
- Combining sentiment with price action
- Building automated alert systems

### Additional Resources

- **Sentiment Analysis in Finance:** Academic research on news impact
- **Time Series Weighting:** Exponential smoothing techniques
- **Source Credibility:** Media bias and reliability metrics

---

**Next:** [Module 3, Notebook 3 - Signal Pipeline](03_signal_pipeline.ipynb)